# TCRnet vs TCRempNet: conservative paired-chain validation

This notebook compares TCRnet and the selected TCRempNet/RedCEA preset in the conservative `alpha AND beta` validation mode, without VJ fixing for both validation epitopes.

Selected TCRempNet preset:

- TRA: `k=10`, `resolution=3`, `min_samples=3`
- TRB: `k=5`, `resolution=2.5`, `min_samples=3`
- validation rule: `alpha AND beta`
- VJ fix: disabled

In [ ]:
from pathlib import Path

import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent

METRICS_PATH = ROOT / 'results' / 'hyperparam_opt' / 'combined_zip_plus_results_folder' / 'tcrnet_vs_tcrempnet_one_preset_no_vj.csv'
FIGURE_DIR = ROOT / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#e6e6e6',
    'grid.linewidth': 0.8,
    'grid.alpha': 1.0,
})

COLORS = {
    'TCRnet cluster_members.txt': '#4c78a8',
    'TCRempNet one preset': '#54a24b',
}
METHOD_LABELS = {
    'TCRnet cluster_members.txt': 'TCRnet',
    'TCRempNet one preset': 'TCRempNet',
}
METRIC_LABELS = {
    'precision': 'Precision',
    'recall': 'Recall',
    'f1': 'F1',
}
EPITOPE_ORDER = ['YLQ', 'GLC']
METHOD_ORDER = ['TCRnet cluster_members.txt', 'TCRempNet one preset']

In [ ]:
metrics = pd.read_csv(METRICS_PATH)
and_metrics = (
    metrics.query("mode == 'and' and with_vj == False")
    .copy()
    .assign(
        method_label=lambda df: df['method'].map(METHOD_LABELS),
        epitope_short=lambda df: pd.Categorical(df['epitope_short'], categories=EPITOPE_ORDER, ordered=True),
    )
    .sort_values(['epitope_short', 'method_label'])
)

display_cols = ['method_label', 'epitope_short', 'precision', 'recall', 'f1', 'true_positive', 'false_positive', 'false_negative']
and_metrics[display_cols].round(3)

In [ ]:
def save_figure(fig: mpl.figure.Figure, stem: str) -> None:
    png = FIGURE_DIR / f'{stem}.png'
    pdf = FIGURE_DIR / f'{stem}.pdf'
    fig.savefig(png, bbox_inches='tight')
    fig.savefig(pdf, bbox_inches='tight')
    print(f'Saved {png}')
    print(f'Saved {pdf}')


def add_value_labels(ax: mpl.axes.Axes, bars, fmt: str = '{:.2f}', dy: float = 0.018) -> None:
    for bar in bars:
        value = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + dy,
            fmt.format(value),
            ha='center',
            va='bottom',
            fontsize=8,
        )

## Main comparison figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.8, 3.8), sharey=True, constrained_layout=True)
x = np.arange(len(EPITOPE_ORDER))
width = 0.34

for ax, metric in zip(axes, ['precision', 'recall', 'f1']):
    for idx, method in enumerate(METHOD_ORDER):
        values = [
            float(and_metrics.loc[(and_metrics['method'].eq(method)) & (and_metrics['epitope_short'].eq(epitope)), metric].iloc[0])
            for epitope in EPITOPE_ORDER
        ]
        bars = ax.bar(
            x + (idx - 0.5) * width,
            values,
            width=width,
            color=COLORS[method],
            label=METHOD_LABELS[method],
            edgecolor='white',
            linewidth=0.8,
        )
        add_value_labels(ax, bars)
    ax.set_title(METRIC_LABELS[metric])
    ax.set_xticks(x)
    ax.set_xticklabels(EPITOPE_ORDER)
    ax.set_ylim(0, 1.05)
    ax.set_axisbelow(True)
    ax.grid(axis='y')
    ax.grid(axis='x', visible=False)
axes[0].set_ylabel('Metric value')
axes[0].legend(frameon=False, loc='lower left')
fig.suptitle('Conservative paired-chain validation: alpha AND beta, no VJ fix', fontsize=13, fontweight='bold')
save_figure(fig, 'tcrnet_tcrempnet_and_no_vj_metric_bars')
plt.show()

## TCRempNet minus TCRnet delta

In [ ]:
delta_rows = []
for epitope in EPITOPE_ORDER:
    tcr = and_metrics[(and_metrics['method'].eq('TCRnet cluster_members.txt')) & (and_metrics['epitope_short'].eq(epitope))].iloc[0]
    tcremp = and_metrics[(and_metrics['method'].eq('TCRempNet one preset')) & (and_metrics['epitope_short'].eq(epitope))].iloc[0]
    for metric in ['precision', 'recall', 'f1']:
        delta_rows.append({
            'epitope_short': epitope,
            'metric': metric,
            'delta': float(tcremp[metric] - tcr[metric]),
        })
delta_df = pd.DataFrame(delta_rows)
delta_df

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.9), constrained_layout=True)
labels = [METRIC_LABELS[m] for m in ['precision', 'recall', 'f1']]
x = np.arange(len(labels))
width = 0.34
delta_colors = {'YLQ': '#6f6f6f', 'GLC': '#2f7f68'}

for idx, epitope in enumerate(EPITOPE_ORDER):
    vals = [
        float(delta_df.loc[(delta_df['epitope_short'].eq(epitope)) & (delta_df['metric'].eq(metric)), 'delta'].iloc[0])
        for metric in ['precision', 'recall', 'f1']
    ]
    bars = ax.bar(
        x + (idx - 0.5) * width,
        vals,
        width=width,
        color=delta_colors[epitope],
        label=epitope,
        edgecolor='white',
        linewidth=0.8,
    )
    for bar, value in zip(bars, vals):
        va = 'bottom' if value >= 0 else 'top'
        offset = 0.018 if value >= 0 else -0.018
        ax.text(bar.get_x() + bar.get_width() / 2, value + offset, f'{value:+.2f}', ha='center', va=va, fontsize=8)

ax.axhline(0, color='#333333', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('TCRempNet - TCRnet')
ax.set_title('Metric changes in conservative validation mode', fontweight='bold')
ax.legend(frameon=False, ncol=2, loc='upper left')
ax.grid(axis='y')
ax.grid(axis='x', visible=False)
save_figure(fig, 'tcrnet_tcrempnet_and_no_vj_delta_bars')
plt.show()

## Confusion-count summary

In [ ]:
count_cols = ['true_positive', 'false_positive', 'false_negative']
count_labels = {'true_positive': 'TP', 'false_positive': 'FP', 'false_negative': 'FN'}

fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.7), sharey=True, constrained_layout=True)
x = np.arange(len(count_cols))
width = 0.34

for ax, epitope in zip(axes, EPITOPE_ORDER):
    ep_df = and_metrics[and_metrics['epitope_short'].eq(epitope)]
    for idx, method in enumerate(METHOD_ORDER):
        row = ep_df[ep_df['method'].eq(method)].iloc[0]
        values = [int(row[col]) for col in count_cols]
        bars = ax.bar(
            x + (idx - 0.5) * width,
            values,
            width=width,
            color=COLORS[method],
            label=METHOD_LABELS[method],
            edgecolor='white',
            linewidth=0.8,
        )
        for bar in bars:
            value = int(bar.get_height())
            ax.text(bar.get_x() + bar.get_width() / 2, value + 2, str(value), ha='center', va='bottom', fontsize=8)
    ax.set_title(epitope)
    ax.set_xticks(x)
    ax.set_xticklabels([count_labels[col] for col in count_cols])
    ax.grid(axis='y')
    ax.grid(axis='x', visible=False)
axes[0].set_ylabel('Number of TCRvdb rows')
axes[0].legend(frameon=False, loc='upper right')
fig.suptitle('Confusion counts: alpha AND beta, no VJ fix', fontsize=13, fontweight='bold')
save_figure(fig, 'tcrnet_tcrempnet_and_no_vj_confusion_bars')
plt.show()

## Compact table figure

In [ ]:
table_df = and_metrics.copy()
table_df['Method'] = table_df['method'].map(METHOD_LABELS)
table_df = table_df[['Method', 'epitope_short', 'precision', 'recall', 'f1']]
table_df = table_df.pivot(index='Method', columns='epitope_short', values=['precision', 'recall', 'f1'])
table_df = table_df.reindex(['TCRnet', 'TCRempNet'])
table_df = table_df.loc[:, pd.MultiIndex.from_product([['precision', 'recall', 'f1'], EPITOPE_ORDER])]
table_df.columns = [f'{METRIC_LABELS[m]} {e}' for m, e in table_df.columns]
display(table_df.round(3))

fig, ax = plt.subplots(figsize=(9.5, 1.8), constrained_layout=True)
ax.axis('off')
cell_text = table_df.round(3).astype(str).values.tolist()
table = ax.table(
    cellText=cell_text,
    rowLabels=table_df.index.tolist(),
    colLabels=table_df.columns.tolist(),
    loc='center',
    cellLoc='center',
    rowLoc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1, 1.35)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('#c8c8c8')
    if row == 0:
        cell.set_facecolor('#f2f2f2')
        cell.set_text_props(weight='bold')
    if col == -1:
        cell.set_text_props(weight='bold')
        if row == 1:
            cell.set_facecolor('#eaf0f7')
        elif row == 2:
            cell.set_facecolor('#eaf5e8')
ax.set_title('Conservative validation metrics (alpha AND beta, no VJ fix)', pad=8, fontweight='bold')
save_figure(fig, 'tcrnet_tcrempnet_and_no_vj_table')
plt.show()

## Clustering coverage across all epitopes

This section compares the number of clustered points and cluster-size distributions across all shared epitopes, using:

- TCRnet: `results/tcrnet/cluster_members.txt`
- RedCEA/TCRempNet: `results/redcea/cluster_members_TRA.txt` and `results/redcea/cluster_members_TRB.txt`

Counts are computed separately for TRA and TRB and then shown on common epitopes present in both methods.


In [ ]:
TCRNET_CLUSTER_PATH = ROOT / 'results' / 'tcrnet' / 'cluster_members.txt'
REDCEA_TRA_PATH = ROOT / 'results' / 'redcea' / 'cluster_members_TRA.txt'
REDCEA_TRB_PATH = ROOT / 'results' / 'redcea' / 'cluster_members_TRB.txt'

CLUSTER_KEY = ['species', 'gene', 'antigen.epitope', 'cdr3aa', 'v.segm', 'j.segm', 'cid']
POINT_KEY = ['species', 'gene', 'antigen.epitope', 'cdr3aa', 'v.segm', 'j.segm']


def load_cluster_table(path: Path, method: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep='\t').copy()
    df = df[df['species'].eq('HomoSapiens')].copy()
    df['method'] = method
    df['cdr3aa'] = df['cdr3aa'].fillna('').astype(str)
    df['v.segm'] = df['v.segm'].fillna('').astype(str).str.split(',').str[0].str.strip().str.split('*').str[0]
    df['j.segm'] = df['j.segm'].fillna('').astype(str).str.split(',').str[0].str.strip().str.split('*').str[0]
    df['cid'] = df['cid'].fillna('').astype(str)
    return df.drop_duplicates(subset=CLUSTER_KEY).reset_index(drop=True)


def load_redcea_all() -> pd.DataFrame:
    tra = load_cluster_table(REDCEA_TRA_PATH, 'RedCEA')
    trb = load_cluster_table(REDCEA_TRB_PATH, 'RedCEA')
    return pd.concat([tra, trb], ignore_index=True)


tcrnet_all = load_cluster_table(TCRNET_CLUSTER_PATH, 'TCRnet')
redcea_all = load_redcea_all()

print('TCRnet rows:', tcrnet_all.shape)
print('RedCEA rows:', redcea_all.shape)
print('TCRnet chains:', sorted(tcrnet_all['gene'].unique()))
print('RedCEA chains:', sorted(redcea_all['gene'].unique()))


In [ ]:
def per_epitope_counts(df, method):
    points = df.drop_duplicates(subset=POINT_KEY).copy()
    out = points.groupby(['gene', 'antigen.epitope']).size().reset_index(name='clustered_points')
    out['method'] = method
    return out


def cluster_size_stats(df, method):
    cluster_sizes = (
        df.drop_duplicates(subset=POINT_KEY + ['cid'])
        .groupby(['gene', 'antigen.epitope', 'cid'])
        .size()
        .reset_index(name='cluster_size')
    )
    cluster_sizes['method'] = method
    stats = (
        cluster_sizes.groupby(['gene', 'antigen.epitope'])['cluster_size']
        .agg(['count', 'mean', 'median', 'max'])
        .reset_index()
        .rename(columns={
            'count': 'n_clusters',
            'mean': 'mean_cluster_size',
            'median': 'median_cluster_size',
            'max': 'max_cluster_size',
        })
    )
    stats['method'] = method
    return cluster_sizes, stats


tcr_counts = per_epitope_counts(tcrnet_all, 'TCRnet')
red_counts = per_epitope_counts(redcea_all, 'RedCEA')
tcr_cluster_sizes, tcr_size_stats = cluster_size_stats(tcrnet_all, 'TCRnet')
red_cluster_sizes, red_size_stats = cluster_size_stats(redcea_all, 'RedCEA')

coverage = (
    tcr_counts.rename(columns={'clustered_points': 'tcrnet_clustered_points'})
    .merge(
        red_counts.rename(columns={'clustered_points': 'redcea_clustered_points'}),
        on=['gene', 'antigen.epitope'],
        how='inner',
    )
)
coverage['redcea_minus_tcrnet'] = coverage['redcea_clustered_points'] - coverage['tcrnet_clustered_points']
coverage['redcea_to_tcrnet_ratio'] = coverage['redcea_clustered_points'] / coverage['tcrnet_clustered_points']

size_summary = (
    tcr_size_stats.rename(columns={
        'n_clusters': 'tcrnet_n_clusters',
        'mean_cluster_size': 'tcrnet_mean_cluster_size',
        'median_cluster_size': 'tcrnet_median_cluster_size',
        'max_cluster_size': 'tcrnet_max_cluster_size',
    })
    .merge(
        red_size_stats.rename(columns={
            'n_clusters': 'redcea_n_clusters',
            'mean_cluster_size': 'redcea_mean_cluster_size',
            'median_cluster_size': 'redcea_median_cluster_size',
            'max_cluster_size': 'redcea_max_cluster_size',
        }),
        on=['gene', 'antigen.epitope'],
        how='inner',
    )
)

coverage_out = ROOT / 'results' / 'hyperparam_opt' / 'combined_zip_plus_results_folder' / 'tcrnet_redcea_all_epitope_clustered_counts.csv'
size_out = ROOT / 'results' / 'hyperparam_opt' / 'combined_zip_plus_results_folder' / 'tcrnet_redcea_all_epitope_cluster_size_summary.csv'
coverage.to_csv(coverage_out, index=False)
size_summary.to_csv(size_out, index=False)
print(f'Saved {coverage_out}')
print(f'Saved {size_out}')

display(coverage.sort_values(['gene', 'redcea_clustered_points'], ascending=[True, False]).head(12))
display(size_summary.sort_values(['gene', 'redcea_mean_cluster_size'], ascending=[True, False]).head(12))




In [ ]:
method_totals = pd.DataFrame([
    {
        'method': 'TCRnet',
        'clustered_points': int(tcrnet_all.drop_duplicates(subset=POINT_KEY).shape[0]),
        'clusters': int(tcrnet_all[['gene', 'antigen.epitope', 'cid']].drop_duplicates().shape[0]),
        'mean_cluster_size': float(tcr_cluster_sizes['cluster_size'].mean()),
        'median_cluster_size': float(tcr_cluster_sizes['cluster_size'].median()),
    },
    {
        'method': 'RedCEA',
        'clustered_points': int(redcea_all.drop_duplicates(subset=POINT_KEY).shape[0]),
        'clusters': int(redcea_all[['gene', 'antigen.epitope', 'cid']].drop_duplicates().shape[0]),
        'mean_cluster_size': float(red_cluster_sizes['cluster_size'].mean()),
        'median_cluster_size': float(red_cluster_sizes['cluster_size'].median()),
    },
])
method_totals


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.6), constrained_layout=True)
summary_colors = {'TCRnet': COLORS['TCRnet cluster_members.txt'], 'RedCEA': COLORS['TCRempNet one preset']}

for ax, metric, title, ylabel in [
    (axes[0], 'clustered_points', 'Clustered points', 'Unique clustered points'),
    (axes[1], 'mean_cluster_size', 'Mean cluster size', 'Cluster members per cluster'),
]:
    bars = ax.bar(
        method_totals['method'],
        method_totals[metric],
        color=[summary_colors[m] for m in method_totals['method']],
        edgecolor='white',
        linewidth=0.8,
    )
    for bar in bars:
        value = bar.get_height()
        label = f'{value:,.0f}' if metric == 'clustered_points' else f'{value:.1f}'
        ax.text(bar.get_x() + bar.get_width() / 2, value * 1.01, label, ha='center', va='bottom', fontsize=9)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y')
    ax.grid(axis='x', visible=False)

fig.suptitle('All-epitope clustering coverage and average cluster size', fontsize=13, fontweight='bold')
save_figure(fig, 'tcrnet_redcea_all_epitope_coverage_size_bars')
plt.show()


In [ ]:
def plot_clustered_count_scatter(counts, chain, stem):
    data = counts.copy() if chain is None else counts[counts['gene'].eq(chain)].copy()
    title_chain = 'TRA + TRB' if chain is None else chain
    fig, ax = plt.subplots(figsize=(5.5, 5.0), constrained_layout=True)
    ax.scatter(
        data['tcrnet_clustered_points'],
        data['redcea_clustered_points'],
        s=52,
        color='#54a24b',
        alpha=0.82,
        edgecolors='white',
        linewidths=0.7,
    )
    max_value = int(max(data['tcrnet_clustered_points'].max(), data['redcea_clustered_points'].max()))
    lim = max_value * 1.08
    ax.plot([0, lim], [0, lim], color='#777777', linestyle='--', linewidth=1.0)
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('TCRnet clustered points')
    ax.set_ylabel('RedCEA clustered points')
    ax.set_title(f'Clustered points per epitope: {title_chain}', fontweight='bold')
    ax.grid(color='#e6e6e6')

    label_df = data.assign(total=lambda df: df[['tcrnet_clustered_points', 'redcea_clustered_points']].max(axis=1))
    label_df = label_df.sort_values('total', ascending=False).head(8)
    for _, row in label_df.iterrows():
        ax.annotate(
            row['antigen.epitope'],
            (row['tcrnet_clustered_points'], row['redcea_clustered_points']),
            xytext=(4, 4),
            textcoords='offset points',
            fontsize=7,
            color='#333333',
        )
    save_figure(fig, stem)
    plt.show()


plot_clustered_count_scatter(coverage, 'TRA', 'tcrnet_redcea_clustered_count_scatter_TRA')
plot_clustered_count_scatter(coverage, 'TRB', 'tcrnet_redcea_clustered_count_scatter_TRB')
plot_clustered_count_scatter(coverage, None, 'tcrnet_redcea_clustered_count_scatter_all_chains')


In [ ]:
fig, ax = plt.subplots(figsize=(7.3, 4.2), constrained_layout=True)
cluster_size_df = pd.concat([
    tcr_cluster_sizes[['method', 'gene', 'cluster_size']],
    red_cluster_sizes[['method', 'gene', 'cluster_size']],
], ignore_index=True)
cluster_size_df['method_label'] = cluster_size_df['method'].replace({'RedCEA': 'RedCEA', 'TCRnet': 'TCRnet'})
positions = []
labels = []
box_data = []
colors = []
position = 1
for gene in ['TRA', 'TRB']:
    for method in ['TCRnet', 'RedCEA']:
        vals = cluster_size_df[(cluster_size_df['gene'].eq(gene)) & (cluster_size_df['method'].eq(method))]['cluster_size'].to_numpy()
        if len(vals) == 0:
            continue
        box_data.append(vals)
        positions.append(position)
        labels.append(f'{gene} - {method}')
        colors.append(summary_colors[method])
        position += 1
    position += 0.6

bp = ax.boxplot(box_data, positions=positions, widths=0.55, patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.78)
    patch.set_edgecolor('#333333')
for element in ['whiskers', 'caps', 'medians']:
    for artist in bp[element]:
        artist.set_color('#333333')
ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.set_ylabel('Cluster size')
ax.set_title('Cluster-size distribution across all epitopes', fontweight='bold')
ax.grid(axis='y')
ax.grid(axis='x', visible=False)
save_figure(fig, 'tcrnet_redcea_all_epitope_cluster_size_boxplot')
plt.show()



## Final one-panel comparison

One consolidated panel combines validation metrics, clustered-point coverage, cluster-size summaries, and all-epitope scatterplots.


In [ ]:
# final_one_panel_summary
# One consolidated figure for the manuscript draft.

from pathlib import Path

import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.text import Text
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent
FIGURE_DIR = ROOT / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = ROOT / 'results' / 'hyperparam_opt' / 'combined_zip_plus_results_folder' / 'tcrnet_vs_tcrempnet_one_preset_no_vj.csv'
COVERAGE_PATH = ROOT / 'results' / 'hyperparam_opt' / 'combined_zip_plus_results_folder' / 'tcrnet_redcea_all_epitope_clustered_counts.csv'
VDJDB_SLIM_PATH = ROOT / 'vdjdb_release' / 'vdjdb.slim.txt'
TCRNET_CLUSTER_PATH = ROOT / 'results' / 'tcrnet' / 'cluster_members.txt'
REDCEA_TRA_PATH = ROOT / 'results' / 'redcea' / 'cluster_members_TRA.txt'
REDCEA_TRB_PATH = ROOT / 'results' / 'redcea' / 'cluster_members_TRB.txt'

COLORS = {
    'TCRnet cluster_members.txt': '#4c78a8',
    'TCRempNet one preset': '#f58518',
    'TCRnet': '#4c78a8',
    'RedCEA': '#f58518',
    'VDJdb': '#9e9e9e',
}
METHOD_LABELS = {
    'TCRnet cluster_members.txt': 'TCRnet',
    'TCRempNet one preset': 'RedCEA',
}
METHOD_ORDER = ['TCRnet cluster_members.txt', 'TCRempNet one preset']
EPITOPE_ORDER = ['YLQ', 'GLC']
METRIC_LABELS = {'precision': 'Precision', 'recall': 'Recall', 'f1': 'F1', 'odds_ratio': 'Odds ratio'}
POINT_KEY = ['species', 'gene', 'antigen.epitope', 'cdr3aa', 'v.segm', 'j.segm']


def save_figure(fig, stem):
    png = FIGURE_DIR / (stem + '.png')
    pdf = FIGURE_DIR / (stem + '.pdf')
    fig.savefig(png, bbox_inches='tight')
    fig.savefig(pdf, bbox_inches='tight')
    print('Saved {}'.format(png))
    print('Saved {}'.format(pdf))


def normalize_segment(value):
    if pd.isna(value):
        return value
    value = str(value).split(',')[0].strip()
    return value.split('*')[0]


def load_cluster_table(path, method):
    df = pd.read_csv(path, sep='\t').copy()
    df['method'] = method
    if 'species' in df.columns:
        df = df[df['species'].eq('HomoSapiens')].copy()
    for col in ['v.segm', 'j.segm']:
        if col in df.columns:
            df[col] = df[col].map(normalize_segment)
    return df


def cluster_sizes_from_table(df, method):
    sizes = (
        df.drop_duplicates(subset=POINT_KEY + ['cid'])
        .groupby(['gene', 'antigen.epitope', 'cid'])
        .size()
        .reset_index(name='cluster_size')
    )
    sizes['method'] = method
    return sizes


def add_panel_label(ax, label, x=-0.12, y=1.08):
    ax.text(x, y, label, transform=ax.transAxes, fontsize=16, fontweight='normal', va='top', ha='left')


def prettify_axis(ax, grid_axis='y'):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color='#d9d9d9', linewidth=0.7, alpha=0.75)
    ax.set_axisbelow(True)


def fmt_int(value):
    return '{:,}'.format(int(value)).replace(',', ' ')


metrics = pd.read_csv(METRICS_PATH)
and_metrics = (
    metrics.query("mode == 'and' and with_vj == False")
    .copy()
    .assign(
        method_label=lambda df: df['method'].map(METHOD_LABELS),
        epitope_short=lambda df: pd.Categorical(df['epitope_short'], categories=EPITOPE_ORDER, ordered=True),
    )
    .sort_values(['epitope_short', 'method_label'])
)
and_metrics['true_negative'] = (
    and_metrics['total']
    - and_metrics['true_positive']
    - and_metrics['false_positive']
    - and_metrics['false_negative']
)
and_metrics['odds_ratio'] = (
    (and_metrics['true_positive'] * and_metrics['true_negative'])
    / (and_metrics['false_positive'] * and_metrics['false_negative'])
).replace([np.inf, -np.inf], np.nan)

coverage = pd.read_csv(COVERAGE_PATH)
coverage = coverage[coverage['gene'].isin(['TRA', 'TRB'])].copy()

# Reproduce the multi-epitope / cross-reactivity plot from clustering_comparison.ipynb.
vdjdb_slim = pd.read_csv(VDJDB_SLIM_PATH, sep='\t')
redcea_for_multi = {
    'TRA': pd.read_csv(REDCEA_TRA_PATH, sep='\t'),
    'TRB': pd.read_csv(REDCEA_TRB_PATH, sep='\t'),
}

def compute_multi_stats(gene):
    redcea_multi_counts = (
        redcea_for_multi[gene][['cdr3aa', 'antigen.epitope']]
        .drop_duplicates()
        .groupby('cdr3aa')
        .size()
    )
    gene_vdjdb = vdjdb_slim[vdjdb_slim['gene'].eq(gene)].copy()
    initial_multi = pd.DataFrame(
        gene_vdjdb[['cdr3', 'antigen.epitope']]
        .drop_duplicates()
        .groupby('cdr3')
        .size()[lambda x: x > 1]
    )
    out = (
        initial_multi.reset_index()
        .rename(columns={0: 'vdjdb_count', 'cdr3': 'cdr3aa'})
        .merge(
            pd.DataFrame(redcea_multi_counts, columns=['redcea_count']).reset_index(),
            how='left',
            on='cdr3aa',
        )
        .fillna(0)
        .astype({'redcea_count': int})
    )
    return out[out['cdr3aa'].apply(lambda x: x[0] == 'C' and x[-1] == 'F')].copy()

multi_stats_tra = compute_multi_stats('TRA')
multi_stats_trb = compute_multi_stats('TRB')

# Reload cluster members here so the panel cell is self-contained.
tcrnet_all = load_cluster_table(TCRNET_CLUSTER_PATH, 'TCRnet')
redcea_all = pd.concat([
    load_cluster_table(REDCEA_TRA_PATH, 'RedCEA'),
    load_cluster_table(REDCEA_TRB_PATH, 'RedCEA'),
], ignore_index=True)

tcr_cluster_sizes = cluster_sizes_from_table(tcrnet_all, 'TCRnet')
red_cluster_sizes = cluster_sizes_from_table(redcea_all, 'RedCEA')
cluster_size_df = pd.concat([tcr_cluster_sizes, red_cluster_sizes], ignore_index=True)

chain_totals = []
vdjdb_raw_counts = vdjdb_slim[vdjdb_slim['species'].eq('HomoSapiens')].groupby('gene').size()
for gene in ['TRA', 'TRB']:
    chain_totals.append({
        'method': 'VDJdb',
        'gene': gene,
        'clustered_points': int(vdjdb_raw_counts.get(gene, 0)),
    })
for method, table in [('TCRnet', tcrnet_all), ('RedCEA', redcea_all)]:
    by_chain = table.drop_duplicates(subset=POINT_KEY).groupby('gene').size()
    for gene in ['TRA', 'TRB']:
        chain_totals.append({
            'method': method,
            'gene': gene,
            'clustered_points': int(by_chain.get(gene, 0)),
        })
chain_totals = pd.DataFrame(chain_totals)

mpl.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.weight': 'normal',
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.titleweight': 'normal',
    'axes.labelsize': 12,
    'axes.labelweight': 'normal',
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'legend.title_fontsize': 12,
    'axes.linewidth': 0.8,
})

fig = plt.figure(figsize=(18.0, 9.8))
gs = fig.add_gridspec(
    2,
    4,
    height_ratios=[1.0, 1.16],
    width_ratios=[1.0, 0.82, 1.0, 1.0],
    hspace=0.08,
    wspace=0.10,
)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[0, 2])
ax_d = fig.add_subplot(gs[0, 3])
ax_e = fig.add_subplot(gs[1, 0:2])
ax_f = fig.add_subplot(gs[1, 2:4])

# A. Raw VDJdb rows and clustered points, separately for alpha and beta chains.
x = np.arange(2)
width = 0.24
count_methods = ['TCRnet', 'RedCEA', 'VDJdb']
for midx, method in enumerate(count_methods):
    vals = [
        int(chain_totals.loc[(chain_totals['method'].eq(method)) & (chain_totals['gene'].eq(gene)), 'clustered_points'].iloc[0])
        for gene in ['TRA', 'TRB']
    ]
    bars = ax_a.bar(x + (midx - 1) * width, vals, width=width, color=COLORS[method], label=method)
    for bar, value in zip(bars, vals):
        ax_a.text(bar.get_x() + bar.get_width() / 2, value * 1.015, fmt_int(value), ha='center', va='bottom', fontsize=12, rotation=0)
ax_a.set_xticks(x)
ax_a.set_xticklabels(['TRA', 'TRB'])
ax_a.set_ylabel('Rows / unique clustered points')
ax_a.set_ylim(0, chain_totals['clustered_points'].max() * 1.20)
ax_a.legend(frameon=False, ncol=1, loc='upper left')
prettify_axis(ax_a)
add_panel_label(ax_a, 'A')

# B. Swarm plot of cluster-size distributions by chain.
swarm_parts = []
for (gene, method), sub in cluster_size_df.groupby(['gene', 'method']):
    sub = sub[sub['cluster_size'].gt(0)].copy()
    cutoff = sub['cluster_size'].quantile(0.95)
    sub = sub[sub['cluster_size'].le(cutoff)].copy()
    sub['log_cluster_size'] = np.log10(sub['cluster_size'])
    swarm_parts.append(sub)
cluster_swarm_df = pd.concat(swarm_parts, ignore_index=True)

sns.swarmplot(
    data=cluster_swarm_df,
    x='gene',
    y='log_cluster_size',
    hue='method',
    order=['TRA', 'TRB'],
    hue_order=['TCRnet', 'RedCEA'],
    dodge=True,
    size=2.1,
    alpha=0.58,
    palette={'TCRnet': COLORS['TCRnet'], 'RedCEA': COLORS['RedCEA']},
    ax=ax_b,
)
for xpos, gene in enumerate(['TRA', 'TRB']):
    for offset, method in [(-0.2, 'TCRnet'), (0.2, 'RedCEA')]:
        vals = cluster_swarm_df[(cluster_swarm_df['gene'].eq(gene)) & (cluster_swarm_df['method'].eq(method))]['log_cluster_size']
        if len(vals):
            ax_b.scatter(xpos + offset, vals.mean(), s=34, color='white', edgecolor='black', zorder=4)
            ax_b.plot([xpos + offset - 0.09, xpos + offset + 0.09], [vals.median(), vals.median()], color='black', linewidth=1.2, zorder=4)
log_ticks = [0, np.log10(3), 1, np.log10(30), 2, np.log10(300)]
log_labels = ['1', '3', '10', '30', '100', '300']
ax_b.set_yticks(log_ticks)
ax_b.set_yticklabels(log_labels)
ax_b.set_ylabel('Members per cluster, log scale')
ax_b.set_xlabel('')
ax_b.set_ylim(0, np.log10(300))
handles, labels = ax_b.get_legend_handles_labels()
ax_b.legend(handles[:2], labels[:2], frameon=False, loc='upper left', ncol=2)
prettify_axis(ax_b)
add_panel_label(ax_b, 'B')

# C/D. Log-log scatterplots with identical x/y scales and square plotting areas.
def scatter_panel(ax, chain, label):
    data = coverage[(coverage['gene'].eq(chain)) & coverage['tcrnet_clustered_points'].gt(0) & coverage['redcea_clustered_points'].gt(0)].copy()
    min_positive = float(data[['tcrnet_clustered_points', 'redcea_clustered_points']].min().min())
    max_positive = float(data[['tcrnet_clustered_points', 'redcea_clustered_points']].max().max())
    lower = max(1, 10 ** np.floor(np.log10(min_positive)))
    upper = 2.5 * (10 ** np.ceil(np.log10(max_positive)))
    ax.scatter(
        data['tcrnet_clustered_points'],
        data['redcea_clustered_points'],
        s=48,
        color='#54a24b',
        alpha=0.82,
        edgecolors='white',
        linewidths=0.7,
    )
    ax.plot([lower, upper], [lower, upper], color='#777777', linewidth=1.0, linestyle='--')
    data['label_score'] = data[['tcrnet_clustered_points', 'redcea_clustered_points']].max(axis=1)
    n_labels = 3 if chain == 'TRA' else 2
    label_data = data[data['label_score'].ge(500)].sort_values('label_score', ascending=False).head(n_labels)
    label_offsets = {
        'TRA': {
            'KLGGALQAK': (8, 12),
            'GLCTFVFTL': (10, -2),
            'NLVPMVATV': (10, -16),
            'ELAGIGILTV': (-62, 10),
            'SLLMWITQV': (-62, -10),
        },
        'TRB': {
            'SLLMWITQV': (12, 22),
            'KLGGALQAK': (-90, 2),
            'GLCTFVFTL': (-96, -18),
            'NLVPMVATV': (12, -18),
            'ELAGIGILTV': (-82, 12),
        },
    }
    default_offsets = [(8, 8), (8, -10), (-62, 8), (-62, -10), (10, 24)]
    for idx, (_, row) in enumerate(label_data.iterrows()):
        offset = label_offsets.get(chain, {}).get(row['antigen.epitope'], default_offsets[idx % len(default_offsets)])
        ax.annotate(
            row['antigen.epitope'],
            (row['tcrnet_clustered_points'], row['redcea_clustered_points']),
            xytext=offset,
            textcoords='offset points',
            fontsize=13,
            alpha=0.86,
        )
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlim(lower, upper)
    ax.set_ylim(lower, upper)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('TCRnet clustered points', fontsize=13)
    ax.set_ylabel('RedCEA clustered points', fontsize=13)
    ax.tick_params(axis='both', labelsize=13)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda val, pos: fmt_int(val) if val >= 1 else ''))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda val, pos: fmt_int(val) if val >= 1 else ''))
    prettify_axis(ax, grid_axis='both')
    add_panel_label(ax, label)

scatter_panel(ax_c, 'TRA', 'C')
scatter_panel(ax_d, 'TRB', 'D')

# E/F. Cross-reactive CDR3 plot from clustering_comparison.ipynb.
def plot_multi_cdr3(ax, stats, gene, label, seed):
    rng = np.random.default_rng(seed)
    plot_df = stats.copy()
    plot_df['vdjdb_jitter'] = plot_df['vdjdb_count'] + rng.uniform(-0.08, 0.08, len(plot_df))
    plot_df['redcea_jitter'] = plot_df['redcea_count'] + rng.uniform(-0.06, 0.06, len(plot_df))
    ax.scatter(
        plot_df['vdjdb_jitter'],
        plot_df['redcea_jitter'],
        alpha=0.35,
        s=35,
        color='#4c78a8',
        edgecolors='none',
    )
    ax.set_xscale('symlog')
    ax.set_xlim(left=1.8)
    xticks = [2, 3, 5, 10, 20, 30, 50, 70, 100]
    xticks = [x for x in xticks if x <= stats['vdjdb_count'].max()]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticks)
    y_counts = stats['redcea_count'].value_counts().sort_index()
    ax.set_yticks(y_counts.index)
    x_text = ax.get_xlim()[1]
    for y, count in y_counts.items():
        ax.text(
            x_text,
            y,
            '  n={}'.format(count),
            va='center',
            ha='left',
            fontsize=12,
            color='dimgray',
        )
    ax.set_xlim(ax.get_xlim()[0], ax.get_xlim()[1] * 1.25)
    ax.set_xlabel('vdjdb_count')
    ax.set_ylabel('redcea_count')
    prettify_axis(ax, grid_axis='both')
    add_panel_label(ax, label)

plot_multi_cdr3(ax_c, multi_stats_tra, 'TRA', 'C', 17)
plot_multi_cdr3(ax_d, multi_stats_trb, 'TRB', 'D', 19)

ylq_panel_img = plt.imread(str(FIGURE_DIR / 'ylq_trb_csar_cluster_change_with_logos_panel.png'))
split_col = ylq_panel_img.shape[1] // 2
panel_pad = max(8, int(ylq_panel_img.shape[1] * 0.01))
ylq_left = ylq_panel_img[:, :split_col - panel_pad]
ylq_right = ylq_panel_img[:, split_col + panel_pad:]
for ax, img, label in [(ax_e, ylq_left, 'E'), (ax_f, ylq_right, 'F')]:
    ax.imshow(img, aspect='auto')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    add_panel_label(ax, label)
fig.subplots_adjust(left=0.05, right=0.99, bottom=0.06, top=0.98, wspace=0.28, hspace=0.22)
save_figure(fig, 'tcrnet_tcrempnet_redcea_final_one_panel')
plt.show()
















![Final one-panel comparison](../figures/tcrnet_tcrempnet_redcea_final_one_panel.png)


In [ ]:
# ylq_csar_trb_cluster_change_panel
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import logomaker
from Bio import pairwise2
from matplotlib.lines import Line2D

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 12,
    'axes.titlesize': 12,
    'axes.labelsize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'legend.title_fontsize': 12,
})

TCRNET_CLUSTER_PATH = Path('results/tcrnet/cluster_members.txt')
REDCEA_TRB_PATH = Path('results/redcea/cluster_members_TRB.txt')
FIGURE_DIR = Path('figures')
FIGURE_DIR.mkdir(exist_ok=True)

EPITOPE = 'YLQPRTFLL'
CHAIN = 'TRB'
PREFIX = 'CSAR'
AA_ORDER = list('ACDEFGHIKLMNPQRSTVWY')
METHOD_COLORS = {'TCRnet': '#4c78a8', 'RedCEA': '#54a24b'}


def load_ylq_csar(path, method):
    df = pd.read_csv(path, sep='\t')
    sub = df[
        df['gene'].eq(CHAIN)
        & df['antigen.epitope'].eq(EPITOPE)
        & df['cdr3aa'].astype(str).str.startswith(PREFIX)
    ].copy()
    sub['method'] = method
    sub['cdr3_len'] = sub['cdr3aa'].str.len()
    sub['cluster_short'] = sub['cid'].astype(str).str.extract(r'([^.]+)$')[0]
    sub['cluster_label'] = 'C' + sub['cluster_short']
    sub['x'] = pd.to_numeric(sub['x'], errors='coerce')
    sub['y'] = pd.to_numeric(sub['y'], errors='coerce')
    sub['csz'] = pd.to_numeric(sub['csz'], errors='coerce')
    return sub.sort_values(['cluster_short', 'cdr3aa']).reset_index(drop=True)


def add_panel_label(ax, label, x=-0.045, y=1.08):
    ax.text(
        x,
        y,
        label,
        transform=ax.transAxes,
        fontsize=13,
        fontweight='bold',
        va='top',
        ha='left',
    )


def style_legend(legend):
    frame = legend.get_frame()
    frame.set_facecolor('white')
    frame.set_edgecolor('#b8b8b8')
    frame.set_linewidth(0.8)
    frame.set_alpha(0.86)


def add_logo_separator(ax, y=0.34):
    ax.plot(
        [0.0, 1.0],
        [y, y],
        transform=ax.transAxes,
        color='#9a9a9a',
        linewidth=0.85,
        alpha=0.85,
        clip_on=False,
        zorder=8,
    )


def short_cluster_palette(df, method):
    labels = sorted(df['cluster_label'].unique(), key=lambda value: int(value[1:]) if value[1:].isdigit() else value)
    if len(labels) == 1:
        return {labels[0]: METHOD_COLORS[method]}
    colors = sns.color_palette('Set2', n_colors=len(labels))
    return dict(zip(labels, colors))


def plot_umap(ax, df, method, panel_label):
    plot_df = df.copy()
    if method == 'RedCEA':
        center = plot_df[['x', 'y']].median()
        dx = plot_df['x'] - center['x']
        dy = plot_df['y'] - center['y']
        dist = np.sqrt(dx ** 2 + dy ** 2)
        outlier_cutoff = dist.quantile(0.90)
        shrink = np.where(dist.gt(outlier_cutoff), 0.72, 1.0)
        rng = np.random.default_rng(7)
        plot_df['plot_x'] = center['x'] + dx * shrink + rng.normal(0, 0.006, len(plot_df))
        plot_df['plot_y'] = center['y'] + dy * shrink + rng.normal(0, 0.006, len(plot_df))
        if len(plot_df) > 1:
            sorted_y = plot_df['plot_y'].sort_values()
            lowest_idx = sorted_y.index[0]
            next_lowest_y = sorted_y.iloc[1]
            plot_df.loc[lowest_idx, 'plot_y'] = next_lowest_y
    else:
        plot_df['plot_x'] = plot_df['x']
        plot_df['plot_y'] = plot_df['y']

    palette = short_cluster_palette(plot_df, method)
    min_len = ylq_csar['cdr3_len'].min()
    max_len = ylq_csar['cdr3_len'].max()
    denom = max(max_len - min_len, 1)
    sizes = 42 + 95 * (plot_df['cdr3_len'] - min_len) / denom

    length_palette = None
    if method == 'RedCEA':
        length_order = sorted(df['cdr3_len'].unique())
        greens = sns.color_palette('Greens', n_colors=len(length_order) + 3)[2:]
        length_palette = dict(zip(length_order, greens))
        for length, sub in plot_df.groupby('cdr3_len'):
            ax.scatter(
                sub['plot_x'],
                sub['plot_y'],
                s=sizes.loc[sub.index],
                color=length_palette[length],
                edgecolors='white',
                linewidths=0.65,
                alpha=0.90,
                label=str(length),
            )
    else:
        for cluster_label, sub in plot_df.groupby('cluster_label'):
            ax.scatter(
                sub['plot_x'],
                sub['plot_y'],
                s=sizes.loc[sub.index],
                color=palette[cluster_label],
                edgecolors='white',
                linewidths=0.65,
                alpha=0.88,
                label=cluster_label,
            )

    for cluster_label, sub in plot_df.groupby('cluster_label'):
        cx = sub['plot_x'].median()
        cy = sub['plot_y'].median()
        n = len(sub)
        ax.annotate(
            f'{cluster_label}\nn={n}',
            xy=(cx, cy),
            xytext=(18, -18) if method == 'RedCEA' else (8, 8),
            textcoords='offset points',
            fontsize=12,
            fontweight='bold',
            color='#222222',
            bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='none', alpha=0.82),
        )

    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    if method == 'TCRnet':
        ax.set_aspect('auto')
    else:
        y_min = plot_df['plot_y'].min()
        y_max = plot_df['plot_y'].max()
        y_span = max(y_max - y_min, 1e-6)
        ax.set_ylim(y_min - 0.75 * y_span, y_max + 0.06 * y_span)
        ax.set_aspect('auto')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(True, alpha=0.18, linewidth=0.7)
    sns.despine(ax=ax)

    if method == 'RedCEA':
        handles = [
            Line2D(
                [0],
                [0],
                marker='o',
                linestyle='None',
                markersize=8,
                markerfacecolor=length_palette[length],
                markeredgecolor='white',
                label=str(length),
            )
            for length in sorted(length_palette)
        ]
        legend = ax.legend(
            handles=handles,
            frameon=True,
            fancybox=True,
            title='CDR3 length',
            loc='upper center',
            bbox_to_anchor=(0.50, 0.98),
            ncol=4,
            borderpad=0.35,
            handletextpad=0.45,
            columnspacing=0.85,
        )
        style_legend(legend)
    else:
        ax.set_ylim(bottom=-1600, top=-600)
        legend = ax.legend(
            frameon=True,
            fancybox=True,
            title='Cluster',
            loc='best',
            borderpad=0.35,
            handletextpad=0.45,
        )
        style_legend(legend)

    if panel_label:
        add_panel_label(ax, panel_label)


def align_to_longest(seqs):
    seqs = list(seqs)
    if not seqs:
        return []
    if len({len(seq) for seq in seqs}) == 1:
        return seqs

    ref = max(seqs, key=lambda seq: (len(seq), seq))
    aligned = []
    for seq in seqs:
        aln_ref, aln_seq, *_ = pairwise2.align.globalms(
            ref,
            seq,
            2,
            -1,
            -5,
            -0.5,
            one_alignment_only=True,
        )[0]
        if '-' in aln_ref:
            aln_seq = ''.join(s for r, s in zip(aln_ref, aln_seq) if r != '-')
        aligned.append(aln_seq)
    return aligned


def padded_alignment(aligned_seqs):
    max_len = max(len(seq) for seq in aligned_seqs)
    return [seq + '-' * (max_len - len(seq)) for seq in aligned_seqs]


def frequency_matrix(aligned_seqs):
    padded = padded_alignment(aligned_seqs)
    rows = []
    for pos in range(len(padded[0])):
        counts = {aa: 0 for aa in AA_ORDER}
        for seq in padded:
            aa = seq[pos]
            if aa in counts:
                counts[aa] += 1
        rows.append({aa: counts[aa] / len(padded) for aa in AA_ORDER})
    mat = pd.DataFrame(rows)
    mat.index = range(1, len(mat) + 1)
    return mat


def gap_fractions(aligned_seqs):
    padded = padded_alignment(aligned_seqs)
    return pd.Series(
        [(sum(seq[pos] == '-' for seq in padded) / len(padded)) for pos in range(len(padded[0]))],
        index=range(1, len(padded[0]) + 1),
    )


def draw_gap_marks(ax, gaps, y=1.08):
    nonzero = gaps[gaps.gt(0)]
    if nonzero.empty:
        return
    for pos, frac in nonzero.items():
        ax.hlines(
            y,
            pos - 0.36,
            pos + 0.36,
            color='#1f1f1f',
            linewidth=2.5 + 3.0 * frac,
            clip_on=False,
        )
        ax.text(
            pos + 0.08,
            y + 0.090,
            '{:.0f}%'.format(frac * 100),
            ha='center',
            va='bottom',
            rotation=45,
            rotation_mode='anchor',
            fontsize=12,
            color='#1f1f1f',
            clip_on=False,
        )


def plot_logo(
    ax,
    seqs,
    title,
    panel_label=None,
    show_gap_marks=False,
    title_fontsize=12,
    tick_labelsize=12,
    show_ylabel=True,
    show_xlabel=False,
    show_title=True,
):
    aligned = align_to_longest(seqs)
    mat = frequency_matrix(aligned)
    gaps = gap_fractions(aligned)
    logo = logomaker.Logo(
        mat,
        ax=ax,
        color_scheme='NajafabadiEtAl2017',
        width=0.90,
        vpad=0.04,
    )
    logo.style_spines(visible=False)
    logo.style_spines(spines=['left', 'bottom'], visible=True, linewidth=0.8)
    ax.set_ylim(0, 1.22 if show_gap_marks and gaps.gt(0).any() else 1.02)
    ax.set_xlim(0.5, len(mat) + 0.5)
    ax.set_ylabel('Frequency' if show_ylabel else '')
    if show_title and title:
        ax.set_title(title, fontsize=title_fontsize)
    ax.set_xticks(range(1, len(mat) + 1))
    ax.tick_params(axis='both', labelsize=tick_labelsize)
    ax.grid(axis='y', alpha=0.18, linewidth=0.7)
    if show_gap_marks:
        draw_gap_marks(ax, gaps)
    if panel_label:
        add_panel_label(ax, panel_label, x=-0.035, y=1.12)
    if show_xlabel:
        ax.set_xlabel('CDR3 position')
    return aligned, gaps


def add_tcrnet_logo_insets(ax, tcrnet_df):
    inset_specs = [
        ('C21', [0.035, 0.055, 0.42, 0.19]),
        ('C24', [0.520, 0.055, 0.42, 0.19]),
    ]
    for cluster_label, bounds in inset_specs:
        sub = tcrnet_df[tcrnet_df['cluster_label'].eq(cluster_label)]
        inset = ax.inset_axes(bounds)
        plot_logo(
            inset,
            sub['cdr3aa'].tolist(),
            '(len=16)',
            panel_label=None,
            show_gap_marks=False,
            title_fontsize=12,
            tick_labelsize=12,
            show_ylabel=False,
            show_title=False,
        )
        inset.set_xlabel('')
        inset.set_facecolor('white')
        inset.patch.set_alpha(0.86)
        for spine in inset.spines.values():
            spine.set_visible(False)
        inset.set_yticks([])
        inset.tick_params(axis='x', labelsize=12, length=2, pad=1)
        inset.tick_params(axis='y', length=0)



def redcea_length_palette(redcea_df):
    length_order = sorted(redcea_df['cdr3_len'].unique())
    greens = sns.color_palette('Greens', n_colors=len(length_order) + 3)[2:]
    return dict(zip(length_order, greens))


def add_redcea_summary_insets(ax, redcea_df):
    logo_inset = ax.inset_axes([0.365, 0.035, 0.62, 0.29])
    plot_logo(
        logo_inset,
        redcea_df['cdr3aa'].tolist(),
        '',
        panel_label=None,
        show_gap_marks=True,
        title_fontsize=12,
        tick_labelsize=12,
        show_ylabel=False,
    )
    logo_inset.set_xlabel('')
    logo_inset.set_facecolor('white')
    logo_inset.patch.set_alpha(0.88)
    for spine in logo_inset.spines.values():
        spine.set_visible(False)
    logo_inset.set_yticks([])
    logo_inset.tick_params(axis='x', labelsize=12, length=2, pad=1)
    logo_inset.tick_params(axis='y', length=0)

    length_counts = redcea_df['cdr3_len'].value_counts().sort_index()
    palette = redcea_length_palette(redcea_df)
    dist_inset = ax.inset_axes([0.055, 0.075, 0.26, 0.22])
    x = np.arange(len(length_counts))
    dist_inset.bar(
        x,
        length_counts.values,
        color=[palette[length] for length in length_counts.index],
        edgecolor='white',
        linewidth=0.7,
        alpha=0.94,
    )
    dist_inset.set_xticks(x)
    dist_inset.set_xticklabels(length_counts.index.astype(str))
    for xpos, value in zip(x, length_counts.values):
        dist_inset.text(xpos, value + max(length_counts.values) * 0.035, str(int(value)), ha='center', va='bottom', fontsize=12)
    dist_inset.set_xlabel('aa', fontsize=12, labelpad=1)
    dist_inset.set_ylabel('n', fontsize=12, labelpad=1)
    dist_inset.set_ylim(0, max(length_counts.values) * 1.30)
    dist_inset.tick_params(axis='both', labelsize=12, length=2, pad=1)
    dist_inset.grid(axis='y', alpha=0.22, linewidth=0.5)
    dist_inset.set_facecolor('white')
    dist_inset.patch.set_alpha(0.88)
    sns.despine(ax=dist_inset)


tcrnet_csar = load_ylq_csar(TCRNET_CLUSTER_PATH, 'TCRnet')
redcea_csar = load_ylq_csar(REDCEA_TRB_PATH, 'RedCEA')
ylq_csar = pd.concat([tcrnet_csar, redcea_csar], ignore_index=True)

summary_rows = []
for (method, cid, cluster_label), sub in ylq_csar.groupby(['method', 'cid', 'cluster_label']):
    summary_rows.append({
        'method': method,
        'cid': cid,
        'cluster': cluster_label,
        'n_selected': len(sub),
        'cluster_size': int(sub['csz'].dropna().iloc[0]) if sub['csz'].notna().any() else len(sub),
        'len_min': int(sub['cdr3_len'].min()),
        'len_median': float(sub['cdr3_len'].median()),
        'len_max': int(sub['cdr3_len'].max()),
    })
summary = pd.DataFrame(summary_rows).sort_values(['method', 'cluster']).reset_index(drop=True)
try:
    display(summary)
except NameError:
    print(summary.to_string(index=False))

fig = plt.figure(figsize=(14.2, 6.6), constrained_layout=True)
gs = fig.add_gridspec(1, 2, wspace=0.08)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])

plot_umap(ax_a, tcrnet_csar, 'TCRnet', None)
add_tcrnet_logo_insets(ax_a, tcrnet_csar)
plot_umap(ax_b, redcea_csar, 'RedCEA', None)
add_redcea_summary_insets(ax_b, redcea_csar)

png_path = FIGURE_DIR / 'ylq_trb_csar_cluster_change_panel.png'
pdf_path = FIGURE_DIR / 'ylq_trb_csar_cluster_change_panel.pdf'
combined_png_path = FIGURE_DIR / 'ylq_trb_csar_cluster_change_with_logos_panel.png'
combined_pdf_path = FIGURE_DIR / 'ylq_trb_csar_cluster_change_with_logos_panel.pdf'
for path in [png_path, combined_png_path]:
    fig.savefig(path, dpi=300, bbox_inches='tight')
for path in [pdf_path, combined_pdf_path]:
    fig.savefig(path, bbox_inches='tight')
print(f'Saved {png_path.resolve()}')
print(f'Saved {pdf_path.resolve()}')
print(f'Saved {combined_png_path.resolve()}')
print(f'Saved {combined_pdf_path.resolve()}')
plt.show()

# Supplementary: RedCEA C84 logos split by raw CDR3 length.
lengths = sorted(redcea_csar['cdr3_len'].unique())
supp_fig, supp_axes = plt.subplots(len(lengths), 1, figsize=(13.5, 7.6), constrained_layout=True)
if len(lengths) == 1:
    supp_axes = [supp_axes]
for idx, (ax, length) in enumerate(zip(supp_axes, lengths)):
    sub = redcea_csar[redcea_csar['cdr3_len'].eq(length)]
    plot_logo(
        ax,
        sub['cdr3aa'].tolist(),
        f'RedCEA C84 length {length}: n={len(sub)}',
        panel_label=chr(ord('A') + idx),
        show_gap_marks=False,
        show_xlabel=False,
    )
supp_axes[-1].set_xlabel('CDR3 position')
supp_fig.suptitle('Supplementary: RedCEA C84 CSAR logos by raw CDR3 length', fontsize=14, fontweight='bold')

supp_png_path = FIGURE_DIR / 'ylq_trb_csar_redcea_length_specific_weblogos_supplementary.png'
supp_pdf_path = FIGURE_DIR / 'ylq_trb_csar_redcea_length_specific_weblogos_supplementary.pdf'
supp_fig.savefig(supp_png_path, dpi=300, bbox_inches='tight')
supp_fig.savefig(supp_pdf_path, bbox_inches='tight')
print(f'Saved {supp_png_path.resolve()}')
print(f'Saved {supp_pdf_path.resolve()}')
plt.show()


In [ ]:
# ylq_csar_trb_cluster_weblogos
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import logomaker
from Bio import pairwise2

TCRNET_CLUSTER_PATH = Path('results/tcrnet/cluster_members.txt')
REDCEA_TRB_PATH = Path('results/redcea/cluster_members_TRB.txt')
FIGURE_DIR = Path('figures')
FIGURE_DIR.mkdir(exist_ok=True)

EPITOPE = 'YLQPRTFLL'
CHAIN = 'TRB'
PREFIX = 'CSAR'
AA_ORDER = list('ACDEFGHIKLMNPQRSTVWY')


def load_ylq_csar(path, method):
    df = pd.read_csv(path, sep='\t')
    sub = df[
        df['gene'].eq(CHAIN)
        & df['antigen.epitope'].eq(EPITOPE)
        & df['cdr3aa'].astype(str).str.startswith(PREFIX)
    ].copy()
    sub['method'] = method
    sub['cdr3_len'] = sub['cdr3aa'].str.len()
    sub['cluster_short'] = sub['cid'].astype(str).str.extract(r'([^.]+)$')[0]
    sub['cluster_label'] = 'C' + sub['cluster_short']
    sub['csz'] = pd.to_numeric(sub['csz'], errors='coerce')
    return sub.sort_values(['cluster_short', 'cdr3aa']).reset_index(drop=True)


def align_to_longest(seqs):
    seqs = list(seqs)
    if not seqs:
        return []
    if len({len(seq) for seq in seqs}) == 1:
        return seqs

    ref = max(seqs, key=lambda seq: (len(seq), seq))
    aligned = []
    for seq in seqs:
        aln_ref, aln_seq, *_ = pairwise2.align.globalms(
            ref,
            seq,
            2,
            -1,
            -5,
            -0.5,
            one_alignment_only=True,
        )[0]
        if '-' in aln_ref:
            # Stable reference frame: omit insertion columns relative to the
            # longest reference rather than shifting every sequence.
            aln_seq = ''.join(s for r, s in zip(aln_ref, aln_seq) if r != '-')
        aligned.append(aln_seq)
    return aligned


def padded_alignment(aligned_seqs):
    max_len = max(len(seq) for seq in aligned_seqs)
    return [seq + '-' * (max_len - len(seq)) for seq in aligned_seqs]


def frequency_matrix(aligned_seqs):
    padded = padded_alignment(aligned_seqs)
    rows = []
    for pos in range(len(padded[0])):
        counts = {aa: 0 for aa in AA_ORDER}
        for seq in padded:
            aa = seq[pos]
            if aa in counts:
                counts[aa] += 1
        rows.append({aa: counts[aa] / len(padded) for aa in AA_ORDER})
    mat = pd.DataFrame(rows)
    mat.index = range(1, len(mat) + 1)
    return mat


def gap_fractions(aligned_seqs):
    padded = padded_alignment(aligned_seqs)
    return pd.Series(
        [(sum(seq[pos] == '-' for seq in padded) / len(padded)) for pos in range(len(padded[0]))],
        index=range(1, len(padded[0]) + 1),
    )


def add_panel_label(ax, label):
    ax.text(
        -0.035,
        1.10,
        label,
        transform=ax.transAxes,
        fontsize=13,
        fontweight='bold',
        va='top',
        ha='left',
    )


def draw_gap_marks(ax, gaps, y=1.08):
    nonzero = gaps[gaps.gt(0)]
    if nonzero.empty:
        return
    for pos, frac in nonzero.items():
        ax.hlines(
            y,
            pos - 0.36,
            pos + 0.36,
            color='#1f1f1f',
            linewidth=2.5 + 3.0 * frac,
            clip_on=False,
        )
        ax.text(
            pos,
            y + 0.025,
            '{:.0f}%'.format(frac * 100),
            ha='center',
            va='bottom',
            fontsize=7,
            color='#1f1f1f',
            clip_on=False,
        )


def plot_logo(ax, seqs, title, panel_label=None, show_gap_marks=False):
    aligned = align_to_longest(seqs)
    mat = frequency_matrix(aligned)
    gaps = gap_fractions(aligned)
    logo = logomaker.Logo(
        mat,
        ax=ax,
        color_scheme='NajafabadiEtAl2017',
        width=0.90,
        vpad=0.04,
    )
    logo.style_spines(visible=False)
    logo.style_spines(spines=['left', 'bottom'], visible=True, linewidth=0.8)
    ax.set_ylim(0, 1.15 if show_gap_marks and gaps.gt(0).any() else 1.02)
    ax.set_xlim(0.5, len(mat) + 0.5)
    ax.set_ylabel('Frequency')
    ax.set_title(title)
    ax.set_xticks(range(1, len(mat) + 1))
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis='y', alpha=0.18, linewidth=0.7)
    if show_gap_marks:
        draw_gap_marks(ax, gaps)
    if panel_label:
        add_panel_label(ax, panel_label)
    return aligned, gaps


tcrnet_csar = load_ylq_csar(TCRNET_CLUSTER_PATH, 'TCRnet')
redcea_csar = load_ylq_csar(REDCEA_TRB_PATH, 'RedCEA')

# Variant 1: common comparison logos. RedCEA is aligned across lengths and gap-heavy
# columns are marked by thick dashes above the logo.
logo_groups = [
    ('TCRnet C21', tcrnet_csar[tcrnet_csar['cluster_label'].eq('C21')], False),
    ('TCRnet C24', tcrnet_csar[tcrnet_csar['cluster_label'].eq('C24')], False),
    ('RedCEA C84, aligned lengths 14-18', redcea_csar[redcea_csar['cluster_label'].eq('C84')], True),
]

alignment_summary = []
fig, axes = plt.subplots(len(logo_groups), 1, figsize=(13.5, 7.8), constrained_layout=True)
for ax, panel_label, (title, sub, show_gaps) in zip(axes, ['A', 'B', 'C'], logo_groups):
    seqs = sub['cdr3aa'].tolist()
    aligned, gaps = plot_logo(
        ax,
        seqs,
        f'{title}: n={len(seqs)}, raw lengths={sorted(sub["cdr3_len"].unique())}',
        panel_label=panel_label,
        show_gap_marks=show_gaps,
    )
    alignment_summary.append({
        'cluster': title,
        'n': len(seqs),
        'raw_lengths': ','.join(map(str, sorted(sub['cdr3_len'].unique()))),
        'aligned_length': len(aligned[0]) if aligned else 0,
        'gap_positions': ','.join(map(str, gaps[gaps.gt(0)].index.tolist())),
    })
axes[-1].set_xlabel('Aligned CDR3 position')
fig.suptitle('YLQ TRB CSAR cluster sequence logos: common aligned view', fontsize=14, fontweight='bold')

alignment_summary = pd.DataFrame(alignment_summary)
try:
    display(alignment_summary)
except NameError:
    print(alignment_summary.to_string(index=False))

png_path = FIGURE_DIR / 'ylq_trb_csar_cluster_weblogos.png'
pdf_path = FIGURE_DIR / 'ylq_trb_csar_cluster_weblogos.pdf'
fig.savefig(png_path, dpi=300, bbox_inches='tight')
fig.savefig(pdf_path, bbox_inches='tight')
print(f'Saved {png_path.resolve()}')
print(f'Saved {pdf_path.resolve()}')
plt.show()

# Variant 2: RedCEA only, one common aligned logo plus separate logos by raw CDR3 length.
redcea_groups = [('RedCEA C84 all lengths, aligned', redcea_csar)]
for length in sorted(redcea_csar['cdr3_len'].unique()):
    redcea_groups.append((f'RedCEA C84 length {length}', redcea_csar[redcea_csar['cdr3_len'].eq(length)]))

fig2, axes2 = plt.subplots(len(redcea_groups), 1, figsize=(13.5, 10.2), constrained_layout=True)
for idx, (ax, (title, sub)) in enumerate(zip(axes2, redcea_groups)):
    seqs = sub['cdr3aa'].tolist()
    show_gaps = idx == 0
    plot_logo(
        ax,
        seqs,
        f'{title}: n={len(seqs)}',
        panel_label=chr(ord('A') + idx),
        show_gap_marks=show_gaps,
    )
axes2[-1].set_xlabel('CDR3 position' if len(redcea_groups) > 1 else 'Aligned CDR3 position')
fig2.suptitle('RedCEA C84 CSAR motif: common aligned logo versus length-specific logos', fontsize=14, fontweight='bold')

png_path2 = FIGURE_DIR / 'ylq_trb_csar_redcea_common_and_length_weblogos.png'
pdf_path2 = FIGURE_DIR / 'ylq_trb_csar_redcea_common_and_length_weblogos.pdf'
fig2.savefig(png_path2, dpi=300, bbox_inches='tight')
fig2.savefig(pdf_path2, bbox_inches='tight')
print(f'Saved {png_path2.resolve()}')
print(f'Saved {pdf_path2.resolve()}')
plt.show()
